In [3]:
### Data Cleaning for Table 3: Ecommerce Sales by Indusrty ###

## Cleaning workbook with multiple sheets, where each sheet contains a table for each industry. ##
## Each table contains EDI and Website sales for years 2009-2019 for businesses with 10+ employees and ##
## sales for years 2014-2019 for all businesses including 0-9 employees ##

import pandas as pd

file_path = "/Users/ambersnelling/Documents/UK E-commerce Project/Datasets/Raw/260126_tbl3EcomSalesByIndustry_raw_v2.xlsx"

### dictionary ###
# load sheets into dictionary of DataFrames where each key = sheet name and each value = DataFrame containing that sheet's raw data
# sheets is a dictionary
sheets = pd.read_excel(
    file_path,
    sheet_name=None, # load all sheets
    header=None # no clean header rows on sheets
)

### cleaning function ###
# define reusable function to clean one inustry at a time
def clean_industry_sheet(df, industry_name):
    # create empty list to store clean rows in long format
    # each row will represent: year|industry|population|channel
    records = []
    # keep track of current population block (10_plus, all_includes_0_9) since tables are stacked vertically
    current_population = None
    # iterate row by row through df
    for _, row in df.iterrows():
        ## detect population headers ##
        if isinstance(row[0], str): # checks if first column contains a string
            if "Businesses with 10 or more employees" in row[0]:
                current_population = "10_plus"
            elif "All businesses including 0 to 9 employees" in row [0]:
                current_population = "all_includes_0_9"
            continue
        # skip rows without a year
        if pd.isna(row[1]):
            continue
        # extract year - convert year column into integer
        year = int(row[1])
        # dictionary to map sales channels - allow unpivoting of wide columns into tidy rows
        channels = {
            "website": row[2],
            "edi": row[3],
            "total": row[4]
        }
        # create tidy records - for each channel, if a value exists, create one tidy row and append to records
        # 1 row with 3 channel columns will be converted into 3 rows with 1 channel column
        for channel, value in channels.items():
            if pd.notna(value):
                records.append({
                    "year": year,
                    "industry": industry_name,
                    "population_basis": current_population,
                    "channel": channel,
                    "sales_gbp_bn": value
                })
    # return clean df for one industry            
    return pd.DataFrame(records)
## apply function to all sheets ##
# empty list to collect cleaned industry dfs
cleaned_frames = []
# loop over every sheet
for industry, df in sheets.items():
    cleaned = clean_industry_sheet(df, industry)
    cleaned_frames.append(cleaned)
    
# stack industries/sheets into one fact table
fact_ecommerce_sales = pd.concat(
    cleaned_frames,
    ignore_index=True
)

## standardise text values to avoid join errors and inconsistent filtering in SQL/Tableau ##
fact_ecommerce_sales["industry"] = fact_ecommerce_sales["industry"].str.lower()
fact_ecommerce_sales["channel"] = fact_ecommerce_sales["channel"].str.lower()

## validation ##
# check for missing values
fact_ecommerce_sales.isna().sum()
# ensure each industry has expected year coverage and no accidental row loss
fact_ecommerce_sales.groupby(["industry", "year"]).size()
## export clean data ##
fact_ecommerce_sales.to_csv(
    "260226_tbl3EcomSalesByIndustry_cleaned_formatted_v1.csv",
    index=False
)



